### Import packages

In [10]:
import os
from openai import OpenAI
import pandas as pd
import json
import tqdm

### Check if environment variable is set for OpenAI API Key

In [11]:
try:
    api_key = os.environ["OPENAI_API_KEY"]
except KeyError as exc:
    raise RuntimeError("Environment variable OPENAI_API_KEY missing.") from exc

### Define which OpenAI model to use

In [12]:
MODEL = 'gpt-5-nano'

### Construct OpenAI client instance

In [ ]:
client = OpenAI(api_key=api_key)

def send_LLM(prompt: str) -> str:
    try:
        # Send chat request
        chat_completion = client.chat.completions.create(
            model = MODEL,
            messages=[
                {"role": "system", "content": "You are a rigorous JSON-only annotator of online irony."},
                {"role": "user", "content": prompt}
            ])
    except KeyError as exc:
        raise RuntimeError("Error") from exc

    return chat_completion.choices[0].message.content

### Define schema for irony as JSON

In [ ]:
IRONY_SCHEMA = {
    "irony": {
        "irony_score": "1-5 integer (1 = not ironic, 5 = very ironic)",
        "confidence_score": "1-5 integer (1 = not confident, 5 = very confident)",
        "justification": "string - explanation based on observed language",
        "irony_markers": ["string"]
    } 
}

def prompt_irony_analysis(thread_title: str, comment: str) -> str:
    schema_str = json.dumps(IRONY_SCHEMA, indent=2)
    return f"""
You are a rigorous JSON-only annotator of online irony.

Your task is to analyze whether a given Reddit comment contains irony.
The comments come from the “r/de” subreddit.
The goal is to annotate comments based on the definition and use the results to provide a foundation for a scientific analysis. 

Definition:  

Irony is a rhetorical device in which a statement deliberately creates a discrepancy between its literal meaning and its intended meaning.  

Analyze the COMMENT under the THREAD TITLE below.

Return ONLY one JSON object that strictly matches this schema:

{schema_str}

THREAD TITLE:
\"\"\"{thread_title}\"\"\"

COMMENT:
\"\"\"{comment}\"\"\"

"""

### Read input data

In [15]:
df = pd.read_csv('reddit_irony_comments.csv')

df.head()

,collected_by,id,thread_title,comment
0,Johan,1,Mehrheit sieht ältere im Vorteil – nicht einma...,Solange alle ihre Tiefkühlpizza und RTL haben ...
1,Johan,2,Mehrheit sieht ältere im Vorteil – nicht einma...,Ich bin schockiert.
2,Johan,3,Mehrheit sieht ältere im Vorteil – nicht einma...,"Würden sich die babyboomer zu Tode arbeiten, b..."
3,Johan,4,Mehrheit sieht ältere im Vorteil – nicht einma...,Auf den Schreck erstmal eine Rentenerhöhung.
4,Johan,5,Mehrheit sieht ältere im Vorteil – nicht einma...,Finde den Generationen-Vertrag an sich schon d...


In [16]:
df = df[:5]

### Perform LLM irony analysis on every comment and save as JSON

In [ ]:
import json

results = []

for _, row in df.iterrows():
    row_id = row["id"]
    thread_title = row["thread_title"]
    comment = row["comment"]

    prompt = prompt_irony_analysis(thread_title, comment)
    llm_answer = send_LLM(prompt)

    # Verify correct output format
    try:
        parsed = json.loads(llm_answer)
        irony_obj = parsed.get("irony", {})
    except json.JSONDecodeError:
        irony_obj = {"raw_response": llm_answer}

    results.append({
        "id": row_id,
        "thread_title": thread_title,
        "comment": comment,
        "irony": irony_obj
    })

# Save concatenated results as JSON array
with open("llm_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# Show in notebook
results[:2]

[{'id': 1,
  'thread_title': 'Mehrheit sieht ältere im Vorteil – nicht einmal ein Drittel der jungen glaubt an faire Rente',
  'comment': 'Solange alle ihre Tiefkühlpizza und RTL haben ist doch alles super!',
  'irony': {'irony_score': '4',
   'confidence_score': '4',
   'justification': "The comment reframes a serious topic about pension fairness into a trivial happiness narrative: 'as long as everyone has their frozen pizza and RTL, everything is great.' This implies the speaker believes the issue is solved by consumer comforts, using hyperbolic praise of mundane pleasures to mock or deride the seriousness of pension fairness. The contrast between the stated happiness ('alles super') and the acknowledged problem signals irony/sarcasm.",
   'sarcasm_markers': ['mocking the seriousness of the issue',
    'contrast between serious topic and trivial pleasures',
    "sarcastic assertion that 'everything is great' in a problematic situation"]}},
 {'id': 2,
  'thread_title': 'Mehrheit sieht